# **Detecção de eixos** de caminhões para inferência de carga máxima - utilizando YOLOv8

### Problema
Imprecisão na emissão de alertas automáticos para auditores sobre veículos que poderiam levantar alertas de incongruências de carga em relação às notas fiscais emitidas. 

### Proposta
Os eixos de um caminhão indicam, juntamente com outros fatores, qual é a carga máxima desde veículo. A ideia é detectar essa informação e, de acordo com regras pré-estabelecidas, inferir uma possível carga máxima para um dado veículo (do tipo **caminhão**) analisado.


### Pipeline

1. Detecção de pneus
    
    1.1. Análise e pré-processamento da database (imagens e informações sobre caminhões registrados pelo sistema de monitoramento de veículos de carga da SEFAZ-PB)

    1.2. Anotação das imagens (820 imagens de caminhão)

    1.3. Organização das anotações e imagens que serão submetidas ao modelo *Deep Learning* para *Visão Computacional*  **YOLOv8** 

    1.4. Treinamento do modelo utilizando os conjuntos de treinamento e teste

    1.5. Avaliação do modelo de detecção de eixos

    1.6. Utilização na prática

2. Clusterização de **Pneus** em **Eixos**

    2.1. Estudo de clusterização

    2.2. Implementação da *função de clusterização* elaborada

    2.3. Testes práticos

# Detecção de Eixos

## Análise e Pré-processamento

In [1]:
# para recarregar automaticamente os módulos editados
%load_ext autoreload
%autoreload 2

In [2]:
# seed para reprodutibilidade
RANDOM_SEED = 42

## Anotação das imagens

## Organização dos dados (Padrão YOLOv8)

O modelo **YOLOv8** pede uma estrutura organizada de uma forma específica para operar de forma previsível. Utilizaremos a estrutura a seguir:

```
data/
    train/
        /images
        /labels
    val/
        /images
        /labels
    test/
        /images
        /labels

custom_dataset.yaml
```
---

Para chegar a esta organização, é necessário:
1.   Dividir o dataset em treino, validação e teste
2.   Distribuir nas pastas

In [30]:
# função que faz os dois passos 
from utils import split_train_val

split_train_val(
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    random_seed=42
)

{'total': 820, 'train': 574, 'val': 164, 'test': 82}

## Treinamento

Importação da bilbioteca que contém a classe YOLO

In [ ]:
%pip install ultralytics

In [3]:
from ultralytics import YOLO, settings

Modelos disponíveis para download na Ultralytics disponíveis na documentação: [yolov8](https://docs.ultralytics.com/models/yolov8/#key-features-of-yolov8) 

Para ajustar as configurações do ultralytics, consultar documentação disponível em [ultralytics](https://docs.ultralytics.com/quickstart/#modifying-settings)

In [ ]:
settings

In [48]:
settings.update({
    'datasets_dir': '/home/aria-unimed/axle-detection-herick/',
})

### Iniciar arquitetura

In [ ]:
# arquitetura padrão, por hora
model = YOLO('yolov8n')
    
# Em breve, alterar arquitetura para buscar ajustar ao problema específico, 
# e não usar a arquitetura genérica

Argumentos (hiperparâmentros): [ultralytics-hyperparams](https://docs.ultralytics.com/modes/train/#musgd-optimizer)

Parâmetros de data augmentation: [ultralytics-data-augmentation](https://docs.ultralytics.com/modes/train/#augmentation-settings-and-hyperparameters)

In [ ]:
model.train(
    data='custom_dataset.yaml',
    epochs=30,
    patience=5, # número de épocas sem melhoria para parar o treinamento
    batch=-1, # usar o batch size padrão, que é o melhor para a arquitetura escolhida
    imgsz=640, # tamanho da imagem de entrada
    workers=8, # número de threads utilizadas
    pretrained=True, # usar pesos pré-treinados para acelerar o treinamento
    resume=False, # não retomar de um treinamento anterior
    box=7.5, # peso para a perda de caixa delimitadora, para dar mais importância à localização precisa dos eixos
    cls=0.5, # peso para a perda de classificação, para dar mais importância à identificação correta dos eixos
    dfl=1.5, # peso para a perda de distribuição de distância, para melhorar a precisão da localização dos eixos
    val=True, # realizar validação durante o treinamento para monitorar o desempenho do modelo
    seed=RANDOM_SEED, # semente para reprodutibilidade
    # augmentations
)

## Teste

In [ ]:
# carrega o melhor modelo treinado
model = YOLO("runs/detect/train4/weights/best.pt")

# roda avaliação no conjunto de TESTE
metrics = model.val(data="custom_dataset.yaml", split="test")

## Avaliação

## Uso prático